# Fine-Tuning BERT for Hotel Review Sentiment Classification

> **Historical / exploratory notebook. The canonical implementation is under `src/`.**
### Đồ Án Môn Học: Trí Tuệ Nhân Tạo (Artificial Intelligence)
**Mô hình:** `google-bert/bert-base-uncased`  
**Thư viện:** Hugging Face Transformers + PyTorch  
**Quy trình:** Huấn luyện chuẩn mực có kiểm soát (AdamW, lr=2e-5, early stopping, evaluation mỗi epoch)  

> *Notebook này được thiết kế độc lập, hỗ trợ tính năng **Run All** trên Google Colab GPU (T4/V100) hoặc máy cá nhân.*

## 1. Kiểm tra phần cứng GPU

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    print("Memory Allocated:", round(torch.cuda.memory_allocated(0)/1024**3, 1), "GB")

## 2. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

## 3. Nạp dữ liệu và Tiền xử lý tối thiểu (Minimal Cleaning)
- Bảo toàn từ ngữ tự nhiên cho BERT (không xóa stopword, không lemmatize).
- Xóa thẻ HTML và chuẩn hóa khoảng trắng.

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Thiết lập random seed cố định để kiểm soát tính tái lập
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Đường dẫn dữ liệu (tự động nhận diện local hoặc tải trên Colab)
DATA_FILE = 'dts_20k_raw.csv'
if not os.path.exists(DATA_FILE):
    if os.path.exists('../../Bài giảng/AI.Code/NLP_Demo/dts_20k_raw.csv'):
        DATA_FILE = '../../Bài giảng/AI.Code/NLP_Demo/dts_20k_raw.csv'
    elif os.path.exists('Bài giảng/AI.Code/NLP_Demo/dts_20k_raw.csv'):
        DATA_FILE = 'Bài giảng/AI.Code/NLP_Demo/dts_20k_raw.csv'

print(f"[*] Đang đọc dữ liệu từ: {DATA_FILE}")
df_raw = pd.read_csv(DATA_FILE)

def clean_text_minimal(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df = df_raw.copy()
df["text"] = df["text"].apply(clean_text_minimal)
df["label"] = df["label"].astype(int)
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(f"[*] Tổng số mẫu hợp lệ: {len(df)} | Phân bố: {df['label'].value_counts().to_dict()}")

## 4. Phân chia dữ liệu Stratified (70% Train, 10% Val, 20% Test, seed=42)

In [ ]:
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
for train_val_idx, test_idx in sss_test.split(df, df["label"]):
    train_val_df = df.iloc[train_val_idx].copy()
    test_df = df.iloc[test_idx].copy()

sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED) # 0.125 * 0.8 = 0.10 tổng
for train_idx, val_idx in sss_val.split(train_val_df, train_val_df["label"]):
    train_df = train_val_df.iloc[train_idx].copy()
    val_df = train_val_df.iloc[val_idx].copy()

print(f"[*] Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 5. Tokenization với Hugging Face `AutoTokenizer`

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128  # Dựa trên EDA (90% câu ngắn hơn 128 từ)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_func(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize_func, batched=True)
val_ds = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize_func, batched=True)
test_ds = Dataset.from_pandas(test_df[["text", "label"]]).map(tokenize_func, batched=True)

print("[*] Dữ liệu sau khi tokenize:", train_ds)

## 6. Khởi tạo mô hình BERT và Cấu hình Huấn Luyện (Training Setup)
- Sử dụng `AutoModelForSequenceClassification` với `num_labels=2`.
- Siêu tham số: `learning_rate=2e-5`, `weight_decay=0.01` (AdamW), `epochs=3`, `batch_size=16`.
- Đánh giá sau mỗi epoch, tự động lưu và phục hồi checkpoint có validation loss tốt nhất.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "macro_f1": f1}

training_args = TrainingArguments(
    output_dir="./bert_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
print("[*] Trainer đã sẵn sàng!")

## 7. Tiến hành Huấn Luyện (Training)

In [ ]:
train_result = trainer.train()
print("[*] Huấn luyện hoàn tất! Kết quả:", train_result)

## 8. Đánh giá trên tập kiểm thử độc lập (Test Set)

In [ ]:
raw_preds = trainer.predict(test_ds)
test_preds = np.argmax(raw_preds.predictions, axis=1)
test_labels = raw_preds.label_ids

acc = accuracy_score(test_labels, test_preds)
f1 = f1_score(test_labels, test_preds, average="macro")
prec = precision_score(test_labels, test_preds, average="macro")
rec = recall_score(test_labels, test_preds, average="macro")

print("="*50)
print("KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH BERT TRÊN TEST SET:")
print(f"Accuracy:        {acc*100:.2f}%")
print(f"Macro Precision: {prec:.4f}")
print(f"Macro Recall:    {rec:.4f}")
print(f"Macro F1-Score:  {f1:.4f}")
print("="*50)
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=["Negative (0)", "Positive (1)"], digits=4))

## 9. Ma trận nhầm lẫn (Confusion Matrix)

In [ ]:
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["Negative (0)", "Positive (1)"],
            yticklabels=["Negative (0)", "Positive (1)"],
            annot_kws={"size": 14, "weight": "bold"})
plt.title("BERT Confusion Matrix", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.tight_layout()
plt.show()

## 10. Trích xuất ca dự đoán sai phục vụ Phân Tích Lỗi (Error Analysis)

In [ ]:
test_df["pred"] = test_preds
errors = test_df[test_df["label"] != test_df["pred"]].copy()
print(f"[*] Tổng số ca dự đoán sai: {len(errors)} / {len(test_df)} ({len(errors)/len(test_df)*100:.2f}%)")
print("\nMột số ví dụ dự đoán sai đặc trưng:")
for idx, row in errors.head(5).iterrows():
    print(f"- True: {row['label']} | Pred: {row['pred']} | Text: {row['text'][:120]}...")

## 11. Kiểm thử suy luận (Inference Test)

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = np.argmax(probs)
    label_name = "Positive" if pred_idx == 1 else "Negative"
    return label_name, probs[pred_idx]

samples = [
    "The room was sparkling clean and the staff was extremely welcoming!",
    "Terrible noise all night from the street. Dirty bathroom and rude receptionist.",
    "Not bad for the price, but the elevator was out of order.",
    "Lower price but not bad service at all."
]

for s in samples:
    lbl, conf = predict_sentiment(s)
    print(f"Review: '{s}' -> {lbl} ({conf*100:.2f}%)")